<a href="https://colab.research.google.com/github/Ikenna1414/transformer-semantic-search-engine/blob/main/Semantic_Search_With_Transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Project: Semantic Search With Transformers

In [1]:
# Run this cell beforehand so you do not see any warnings
import warnings
warnings.filterwarnings('ignore')

In [ ]:
!pip install sentence-transformers faiss-cpu


In [ ]:
pip install arxiv


In [4]:
#Import the libraries
import numpy as np

from sentence_transformers import SentenceTransformer
import faiss

In [5]:
import os
os.listdir()

['.config', 'sample_data']

In [6]:
import pickle

In [7]:
os.getcwd()

'/content'

In [8]:
import arxiv
import json

search = arxiv.Search(
    query="machine learning",
    max_results=1000,
    sort_by=arxiv.SortCriterion.SubmittedDate
)

papers = []

for r in search.results():
    papers.append({
        "title": r.title,
        "summary": r.summary,
        "authors": [{"name": a.name} for a in r.authors],
        "tags": [{"term": c} for c in r.categories]
    })

with open("arxivData.json", "w", encoding="utf-8") as f:
    json.dump(papers, f, indent=2)

print("Saved arxivData.json with", len(papers), "papers")


Saved arxivData.json with 1000 papers


In [9]:
with open("arxivData.json", "r") as f:
    raw = json.load(f)


In [10]:
#retrieve the model
model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [11]:
#Generate or load the embeddings
emb_path = "default_embeddings.pkl"

if os.path.exists(emb_path):
    with open(emb_path, "rb") as f:
        embeddings = pickle.load(f)
else:
    texts = [p["summary"] for p in papers]
    embeddings = model.encode(texts, show_progress_bar=True)
    with open(emb_path, "wb") as f:
        pickle.dump(embeddings, f)

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

In [12]:
# Data Preparation and Helper Methods
texts = [p.get("summary", "") for p in papers]

embeddings = model.encode(texts, show_progress_bar=True)
embeddings = np.asarray(embeddings, dtype=np.float32)

def embed_query(query: str) -> np.ndarray:
    q = model.encode([query])
    return np.asarray(q, dtype=np.float32)

def get_text(idx: int) -> str:
    return texts[idx]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

In [13]:
#set up the index
dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)

In [14]:
#search with a summary
query = "deep learning methods for image classification"

q_emb = embed_query(query)
D, I = index.search(q_emb, k=5)

for rank, idx in enumerate(I[0], 1):
    print(f"{rank}. {get_text(idx)[:300]}...")

1. Pattern recognition and image classification are essential tasks in machine vision. Autonomous vehicles, for example, require being able to collect the complex information contained in a changing environment and classify it in real time. Here, we experimentally demonstrate image classification at mu...
2. Histopathology remains the gold standard for cancer diagnosis because it provides detailed cellular-level assessment of tissue morphology. However, manual histopathological examination is time-consuming, labour-intensive, and subject to inter-observer variability, creating a demand for reliable comp...
3. Automated detection and classification of structural cracks and surface defects is a critical challenge in civil engineering, infrastructure maintenance, and heritage preservation. Recent advances in Computer Vision (CV) and Deep Learning (DL) have significantly improved automatic crack detection. H...
4. The unrestrained proliferation of cells that are malignant in nature is canc

In [15]:
# Search with a prompt
def search(prompt, top_k=5):
    query_embedding = model.encode([prompt]).astype("float32")
    distances, indices = index.search(query_embedding, top_k)

    results = []
    for i, idx in enumerate(indices[0]):
        results.append({
            "rank": i + 1,
            "title": papers[idx]["title"],
            "summary": papers[idx]["summary"],
            "distance": float(distances[0][i])
        })
    return results

In [16]:
prompt = "stochastic optimization methods for deep learning"

results = search(prompt, top_k=5)

for r in results:
    print(f"\nRank {r['rank']}")
    print("Title:", r["title"])
    print("Distance:", round(r["distance"], 4))
    print("Summary:", r["summary"][:400], "...")



Rank 1
Title: A Short Survey of Averaging Techniques in Stochastic Gradient Methods
Distance: 0.7672
Summary: Stochastic gradient methods are among the most widely used algorithms for large-scale optimization and machine learning. A key technique for improving the statistical efficiency and stability of these methods is the use of averaging schemes applied to the sequence of iterates generated during optimization. Starting from the classical work on stochastic approximation, averaging techniques such as P ...

Rank 2
Title: SHANG++: Robust Stochastic Acceleration under Multiplicative Noise
Distance: 0.804
Summary: Under the multiplicative noise scaling (MNS) condition, original Nesterov acceleration is provably sensitive to noise and may diverge when gradient noise overwhelms the signal. In this paper, we develop two accelerated stochastic gradient descent methods by discretizing the Hessian-driven Nesterov accelerated gradient flow. We first derive SHANG, a direct Gauss-Seidel-type d